<img src="../logo.png" alt="Vegeta" width="240">

# Race 04 — Quarry (04_quarry)

an excavated pit: a broad floor, terraced benches around it, a haul ramp spiralling up, spoil heaps on the floor — the open course with the most vertical.

The reference 5-inch racing quad (`races.refquad`: the 6S 2207 drive on its Boreas blade-element table) flies one lap under the race rules (`races.rules`: zones in order, collisions reset to the last checkpoint with a penalty, more than five collisions or 300 s is a DNF). Two simulators, the same course and referee:

- **the mock flyer** (`races.mock`, always): the racing line's speed schedule flown by a kinematic tracker — seconds, headless;
- **MuJoCo** (`races.sim`, with `GENERATE_VIDEO = True`): the rigid body under the geometric controller in the course's MJCF scene, then the recorded trajectory rendered from the scene's cameras (`races.video`, mp4 with ffmpeg).

Every run is exported on its own (`races.export`: time series CSV, events, summary with provenance) under `races/runs/notebook/quarry/`; `05_race_requirements` reads those exports and never reruns a race.

In [ ]:
# The races are fast and nothing in them is mocked: the laps below are the point-mass mock flyer, and with
# GENERATE_VIDEO = True a real MuJoCo lap (in Vegeta this cell switched vegeta.mock on, which patched nothing here).
DEBUG = False

In [ ]:
# parameters
GENERATE_VIDEO = False            # True: fly the lap in MuJoCo too and render its trajectory (mujoco; ffmpeg for mp4; an OpenGL context, MUJOCO_GL=egl)
VIDEO_QUALITY = "screening"       # "screening" (short, small) or "fine" (full length, full size: minutes)
CAMERAS = ("overview", "chase", "onboard")
SEED = 0                          # the mock flyer's random seed (it is deterministic for a given seed)
MOCK = dict(dt=0.02, safety=0.8)  # the mock flyer: step [s], share of the thrust-vector corner speed flown
import sys; sys.path.insert(0, "../..")
from races import autopilot
COURSE = "quarry"
MUJOCO = dict(control_dt=0.004, **autopilot.tuned(COURSE))   # the tuned settings (races/autopilot.json, ./tune_autopilot.sh); run_mujoco's defaults where none
COURSE_PARAMS = {}                # overrides of the course builder's parameters (races.courses.quarry), empty = as designed
print("MuJoCo autopilot:", MUJOCO if len(MUJOCO) > 1 else "run_mujoco's defaults (no tuned settings for this course yet)")

In [ ]:
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image, Video
sys.path.insert(0, "../..")
from races import analysis, courses, export, metrics, mock, refquad, replay, rules
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
C = courses.COURSES[COURSE](**COURSE_PARAMS)
RUNS = {}

## 1. The course

The map (plan and elevation) with the obstacles and the numbered zones (green start, blue gates, orange checkpoints, red finish), each zone's clearance, the racing line's speed schedule, the aircraft.

In [ ]:
display(Markdown(f"### {C.name} — {C.kind}, {C.length():.0f} m, {len(C.zones)} zones\n\n{C.description}.  \n*Recovery:* {C.recovery}.  \n*Parameters:* `{C.params}`"))
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]})
replay.overview(C, ax=axes[0], view="plan"); replay.overview(C, ax=axes[1], view="elevation"); plt.tight_layout(); plt.show()
ck = C.clearance_check()
display(pd.DataFrame({"zone": [z.name for z in C.zones], "kind": [z.kind for z in C.zones], "centre": [z.center for z in C.zones], "half size": [z.half for z in C.zones],
                      "nearest obstacle [m]": ck["nearest obstacle [m]"].values, "clear": ck["ok"].values}).set_index("zone"))
SCHED = mock.schedule(C, safety=MOCK["safety"])
print(f"schedule: {SCHED.time_estimate():.1f} s at up to {SCHED.v.max():.0f} m/s (lateral {SCHED.limits['a_lateral'] / 9.81:.1f} g, brake {SCHED.limits['a_brake'] / 9.81:.1f} g)")
display(pd.Series(refquad.summary(), name="reference quad").to_frame())

## 2. The lap — mock flyer

In [ ]:
t0 = time.perf_counter()
LAP = mock.run_mock(C, seed=SEED, **MOCK)
RUNS["mock"] = export.write_run(C, LAP, simulator="mock", config={"mock": MOCK, "course_params": COURSE_PARAMS}, seed=SEED, timestep=MOCK["dt"], wall_s=time.perf_counter() - t0)
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]})
replay.overview(C, LAP, ax=axes[0], view="plan"); replay.overview(C, LAP, ax=axes[1], view="elevation"); plt.tight_layout(); plt.show()
replay.traces(LAP); plt.show()
display(replay.splits(LAP).round(2).T)
display(pd.Series(LAP["stats"], name="referee (mock)").to_frame())
print("exported:", RUNS["mock"])

## 3. What the lap asked of the aircraft

The lap report (`races.analysis.lap_report`): the path coloured by speed and load factor, the timeline, the regimes, the percentiles while moving, the corners, the vertical content, the power duty, the closest approaches; the manoeuvre metrics (`races.metrics`). In the mock the roll and pitch rates are the thrust vector's slew and the yaw rate the heading rate (estimates); MuJoCo records body rates.

In [ ]:
R = analysis.lap_report(LAP["history"], C, LAP["events"])
k = R["kinematics"]
fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]})
analysis.plot_path_coloured(C, k, ax=axes[0], by="speed", view="plan"); analysis.plot_path_coloured(C, k, ax=axes[1], by="load_g", view="elevation"); plt.tight_layout(); plt.show()
analysis.plot_timeline(k, LAP["events"], title=C.name); plt.show()
display(Markdown("**regimes**")); display(R["regimes"].round(3))
display(Markdown("**percentiles while moving**")); display(R["percentiles"].round(2))
display(Markdown(f"**corners** ({len(R['corners'])})")); display(R["corners"].round(2).head(12))
display(pd.Series(R["vertical"], name="vertical").to_frame().T.round(2)); display(pd.Series(R["power"], name="power duty").to_frame().T.round(2))
display(Markdown("**closest approaches**")); display(R["closest approaches"].round(2).head(5))
display(metrics.table({f"{C.name} (mock)": LAP}).round(2))

## 4. MuJoCo lap and videos (`GENERATE_VIDEO`)

The lap flown in MuJoCo is exported as its own run (time series, events, summary) **before** any rendering, so a rendering failure keeps the simulation data; the failure is printed and written into the run's summary (`artifacts.video_errors`). The videos are the recorded trajectory replayed through the scene cameras.

In [ ]:
if GENERATE_VIDEO:
    from races import sim, video
    t0 = time.perf_counter()
    MJ = sim.run_mujoco(C, **MUJOCO)
    wall = time.perf_counter() - t0
    display(pd.Series({k: v for k, v in MJ["stats"].items() if k != "controller"}, name="referee (MuJoCo)").to_frame())
    VIDEOS, ERRORS = {}, {}
    run_dir = export.write_run(C, MJ, simulator="mujoco", config={"mujoco": MUJOCO, "course_params": COURSE_PARAMS}, seed=None, timestep=MUJOCO["control_dt"], wall_s=wall)
    for cam in CAMERAS:
        try:
            f = video.mujoco_video(C, MJ, run_dir / f"{COURSE}_{cam}.mp4", camera=cam, quality=VIDEO_QUALITY)
            if f is None:
                raise RuntimeError("no mp4 written (ffmpeg missing on the PATH?)")
            VIDEOS[cam] = f
        except Exception as e:
            ERRORS[cam] = f"{type(e).__name__}: {e}"
            print(f"RENDER FAILED — {cam}: {ERRORS[cam]} (the simulation data is kept in {run_dir})")
    s = json.loads((run_dir / "summary.json").read_text())                      # record the videos and the failures in the run
    s["artifacts"]["videos"] = {k: Path(v).name for k, v in VIDEOS.items()}; s["artifacts"]["video_errors"] = ERRORS
    (run_dir / "summary.json").write_text(json.dumps(s, indent=1, ensure_ascii=False))
    RUNS["mujoco"] = run_dir
    for cam, f in VIDEOS.items():
        display(Markdown(f"*{cam}* — `{f}`")); display(Video(str(f), embed=False, width=640))
    print("exported:", run_dir)
else:
    print("GENERATE_VIDEO is off: the mock lap only (set GENERATE_VIDEO = True for the MuJoCo lap and its videos)")

## 5. Exports

In [ ]:
rows = {}
for sim_name, d in RUNS.items():
    s = json.loads((Path(d) / "summary.json").read_text())
    rows[sim_name] = {"status": s["status"], "DNF reason": s["dnf_reason"], "lap time [s]": s["lap_time_s"], "collisions": s["collisions"],
                      "elapsed sim time [s]": round(s["elapsed_sim_time_s"], 2), "wall time [s]": round(s["wall_time_s"], 2),
                      "simulator": s["provenance"]["simulator_version"], "course sha256": s["provenance"]["course_sha256"][:12],
                      "model sha256": s["provenance"]["model_sha256"][:12], "config sha256": s["provenance"]["config_sha256"][:12],
                      "folder": d, "videos": ", ".join(s["artifacts"]["videos"]) or "—", "video errors": "; ".join(f"{k}: {v}" for k, v in s["artifacts"]["video_errors"].items()) or "—"}
display(pd.DataFrame(rows).T)